<a href="https://colab.research.google.com/github/muhammed-jaseef/BookRecommenderSystem/blob/main/Enterprise_Knowledge_Assistant_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

In [ ]:
#os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Step-1: Document Loading

In [ ]:
!pip install langchain --quiet
!pip install langchain langchain-community --quiet
!pip install pypdf --quiet

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader("/content/EMPLOYEE_AGREEMENT.pdf")
pages = loader.load()

In [ ]:
len(pages)

13

In [ ]:
# Analysing the content in the PDF
full_text =""
for i in pages:
  full_text += i.page_content

print("No.of Pages: ", len(pages))
print("No.of Lines: ", len(full_text.split("\n")))
print("No.of Words: ", len(full_text.split()))
print("No.of Characters: ", len(full_text))

No.of Pages:  13
No.of Lines:  369
No.of Words:  4527
No.of Characters:  29540


# Step-2:Split the data into Chunks

In [ ]:
!pip install langchain-text-splitters --quiet

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = text_splitter.split_documents(pages)
len(chunks)

49

In [ ]:
print(chunks[0].page_content)
print("-------------------------------------------------------------------------")
print(chunks[1].page_content)
print("-------------------------------------------------------------------------")
print(chunks[2].page_content)
print("-------------------------------------------------------------------------")
print(chunks[3].page_content)
print("-------------------------------------------------------------------------")
print(chunks[4].page_content)

EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT
EXHIBIT 10.1
EMPLOYEE AGREEMENT
THIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,
Inc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,
Suite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in
its entirety the employee agreement dated August 15, 2014 between Employee and the Company. 
WHEREAS, the Company wishes to procure the services of Employee under the terms and
conditions set forth and Employee wishes to be employed on these terms and conditions.
WHEREAS, the parties to this Employee Agreement wish to enter into a written expression of their
relationship as Employer and Employee.
-------------------------------------------------------------------------
relationship as Employer and Employee.
THEREFORE, in consideration of the agreements contained in this Employee Agreement, the
parties, intending to be legall

# Step-3: Creating embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipython-input-2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
sample_docs = [chunks[i].page_content for i in range(len(chunks))]
embeded_vectors= embeddings.embed_documents(sample_docs)
print(embeded_vectors)
print(len(embeded_vectors))
print(len(embeded_vectors[0]))

[[-0.12861329317092896, 0.024827957153320312, 0.04775499552488327, -0.03860320523381233, -0.005144820548593998, 0.02694024331867695, 0.09570793062448502, 0.002911291318014264, -0.010914823040366173, -0.09505116939544678, 0.08428046107292175, 0.04577774554491043, -0.05009370297193527, 0.017705393955111504, 0.05009031668305397, 0.00820882711559534, -0.027408063411712646, 0.028010670095682144, 0.02769462577998638, -0.01998768001794815, 0.002039222279563546, -0.004055015742778778, -0.08332959562540054, -0.014266357757151127, 0.029657909646630287, -0.06344632059335709, 0.006579009350389242, 0.08078648149967194, -0.03891962766647339, 0.013492775149643421, 0.01770605891942978, 0.009230716153979301, 0.08450543880462646, 0.0028133606538176537, 0.03256915509700775, -0.052922192960977554, -0.06315676122903824, -0.0035557637456804514, -0.042443208396434784, -0.05059310421347618, -0.07282383739948273, 0.0013757727574557066, -0.022895997390151024, -0.0046701147221028805, -0.06965609639883041, -0.009

Here each chunk in the document is converted into vector of 384 dimension

# Step-4: Storing in Vector Stores

In [ ]:
!pip install langchain langchain-community chromadb --quiet

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="vector_db"
)
vector_db.persist()

/tmp/ipython-input-2247472185.py:6: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


# Step-5: Retrieval

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

result=retriever.invoke("What is the policy for insurance?")
print(result)

[Document(metadata={'source': '/content/EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page_label': '3', 'moddate': '2024-04-03T12:51:00+00:00', 'creationdate': '2024-04-03T12:51:00+00:00', 'producer': 'Skia/PDF m123', 'title': 'EMPLOYEE AGREEMENT', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'page': 2}, page_content='Company for its employees, including, without limitation, the Company’s health insurance plan. No amounts\npaid to Employee from an employee benefit plan shall count as compensation due Employee as base salary or\nadditional compensation.  Nothing in this Employee Agreement shall prohibit the Company from modifying\nor terminating any of its employee benefit plans in a manner that does not discriminate between Employee\nand other Company employees.\nARTICLE 7\nTermination of Employment\n7.1. Termination of Employment. Employee’s employment hereunder shall automatically terminate\nupon (i) his 

In [ ]:
for i, doc in enumerate(result):
    print(f"\n--- Returned Chunk {i+1} ---")
    print(doc.page_content)



--- Returned Chunk 1 ---
Company for its employees, including, without limitation, the Company’s health insurance plan. No amounts
paid to Employee from an employee benefit plan shall count as compensation due Employee as base salary or
additional compensation.  Nothing in this Employee Agreement shall prohibit the Company from modifying
or terminating any of its employee benefit plans in a manner that does not discriminate between Employee
and other Company employees.
ARTICLE 7
Termination of Employment
7.1. Termination of Employment. Employee’s employment hereunder shall automatically terminate
upon (i) his death; (ii) Employee voluntarily leaving the employ of the Company; (iii) at the Company’s sole
discretion, upon fifteen (15) days prior written notice to Employee if the Company terminates his

--- Returned Chunk 2 ---
court or arbitrator, as the case may be, is specifically authorized to reform and narrow said covenant or
agreement to the extent necessary to make said reformed 

In [ ]:
for i in range(len(result)):
  print(result[i].metadata)

{'source': '/content/EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page_label': '3', 'moddate': '2024-04-03T12:51:00+00:00', 'creationdate': '2024-04-03T12:51:00+00:00', 'producer': 'Skia/PDF m123', 'title': 'EMPLOYEE AGREEMENT', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'page': 2}
{'page': 6, 'moddate': '2024-04-03T12:51:00+00:00', 'producer': 'Skia/PDF m123', 'page_label': '7', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'source': '/content/EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'}
{'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'title': 'EMPLOYEE AGREEMENT', 'page': 12, 'page_label': '13', 'moddate': '2024-04-03T12:51:00+00:00', 'creation

# Step-5: Connect Retriever → LLM

In [ ]:
!pip install langchain-groq --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.5 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
#!pip install langchain-google-genai --quiet

In [ ]:
#from langchain_google_genai import ChatGoogleGenerativeAI

#llm = ChatGoogleGenerativeAI(
#    model="gemini-2.5-flash",
#    temperature=0
#)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}
    """
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [ ]:
response = rag_chain.invoke("What is the base salary")
print(response.content)

The base salary is $14,583 per month.


# Step-5: Add Memory (Chat History)

In [ ]:
!pip install langchain-classic --quiet

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """You are an HR policy assistant.

    Use the conversation history and the context below to answer the question.

    Conversation History:
    {chat_history}

    Context:
    {context}

    Question:
    {question}
    """
)


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,      # retriever automatically invoked
        "question": RunnablePassthrough(),
        "chat_history": lambda _: "\n".join(
            f"Human: {m.content}" if isinstance(m, HumanMessage) else f"AI: {m.content}"
            for m in memory.chat_memory.messages
        )
    }
    | prompt
    | llm
)

In [ ]:
def ask(question):
    # 5a. Get response from RAG chain
    response = rag_chain.invoke(question)

    # 5b. Save conversation to memory
    memory.chat_memory.add_message(HumanMessage(content=question))
    memory.chat_memory.add_message(AIMessage(content=response.content))

    # 5c. Return answer
    return response.content

In [ ]:
print(ask("What is the base salary"))

According to Article 4.1 of the Employee Agreement, the base salary is $14,583 per month.


In [ ]:
print(ask("Which is depend on any other factors"))

Based on the conversation history and context provided, the base salary of $14,583 per month is a fixed amount that does not depend on any other factors.


In [ ]:
print(ask("Year of experience have any impact on this"))

Based on the conversation history and the context provided, the base salary of $14,583 per month is a fixed amount that does not depend on any other factors, including the year of experience.


In [ ]:
print(ask("what is the procedure for termination"))

According to Article 7 of the Employee Agreement, the procedure for termination is as follows:

1. **Automatic Termination**: Employee's employment shall automatically terminate upon:
   - (i) his death
   - (ii) Employee voluntarily leaving the employ of the Company
   - (iii) at the Company's sole discretion, upon fifteen (15) days prior written notice to Employee if the Company terminates his employment hereunder without "cause"

2. **Termination by Company**: The Company may terminate Employee's employment at its sole discretion, upon:
   - (i) fifteen (15) days prior written notice to Employee if the Company terminates his employment hereunder without "cause"
   - (ii) two (2) days prior written notice to Employee if the Company terminates his employment hereunder for "cause"

   For purposes of this Employee Agreement, "cause" shall include:
   - (i) Employee's willful malfeasance, misfeasance, nonfeasance or gross negligence in connection with the performance of his duties
   - 

**Next Method**

In [ ]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [ ]:
from langchain_classic.chains import ConversationalRetrievalChain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

In [ ]:
print(qa_chain.invoke({"question": "What is the base salary"})['answer'])

The base salary is $14,583 per month.


In [ ]:
print(qa_chain.invoke({"question": "it is depended on any factors?"})['answer'])

Based on the provided Employee Agreement, the base salary of $14,583 per month is not affected by any factors mentioned in the agreement. The base salary is a fixed amount that is payable to the Employee not less frequently than bi-monthly, or as is consistent with the Company's practice for its other employees.


In [ ]:
print(qa_chain.invoke({"question": "will it get incremented by any way?"})['answer'])

There is no information in the provided Employee Agreement that mentions any potential increments to the base salary. The base salary is stated as $14,583 per month, and it is mentioned that it shall be payable not less frequently than bi-monthly, but there is no mention of any potential increases or adjustments to the base salary.


In [ ]:
print(qa_chain.invoke({"question": "where is the employemnt lcation"})['answer'])

According to Article 3 of the Employee Agreement, the employment location is 802 NW 5th Avenue, Suite 100, Gainesville, FL 32601.


In [ ]:
print(qa_chain.invoke({"question": "which country is this"})['answer'])

The employment location is in the United States, specifically in Gainesville, Florida. The address is 802 NW 5th Avenue, Suite 100, Gainesville FL 32601.


In [ ]:
print(qa_chain.invoke({"question": "who is the president of India"})['answer'])

I don't know. The provided context is an employee agreement between a company and an employee, and it does not mention the president of India.
